# 📜 Transformer 架构完整解析 — 基于 "Attention Is All You Need"

**论文**: [Attention Is All You Need](https://arxiv.org/abs/1706.03762) (Vaswani et al., NeurIPS 2017)

**本文目标**：逐层拆解 Transformer 的完整架构——从 Self-Attention 的数学本质到 Encoder-Decoder 的每一个子层，结合原始论文中的设计决策和实际代码。读完这篇你会理解：

- Self-Attention 的数学推导和直观含义
- Multi-Head Attention 为什么有效
- Positional Encoding 的设计原理
- Encoder 和 Decoder 的完整数据流
- 论文中的训练细节（Optimizer、LR Schedule、Regularization）
- 这套架构与后续推理优化（KV Cache、FlashAttention 等）的关系

> 本文定位为后续所有推理框架文章的**理论根基**。建议在阅读 `01-transformer-inference-primer.ipynb`（推理视角的 Transformer）之前先读本文。

## 1. 为什么需要 Transformer？— 论文的动机

### 1.1 RNN/LSTM 的致命缺陷

在 Transformer 之前，序列建模的主流方案是 RNN 及其变体（LSTM、GRU）：

```
RNN 的计算方式（串行）:
  h₁ = f(x₁, h₀)
  h₂ = f(x₂, h₁)  ← 必须等 h₁ 算完!
  h₃ = f(x₃, h₂)  ← 必须等 h₂ 算完!
  ...
  hₙ = f(xₙ, hₙ₋₁) ← 必须等 hₙ₋₁ 算完!

三个致命问题:
  1. 无法并行: 序列长度 1000 → 必须串行 1000 步
  2. 长程依赖: 位置 1 的信息要经过 t-1 步传到位置 t → 梯度消失/爆炸
  3. 位置偏差: 相邻位置天然"更亲近"，远距离关系需要更多层
```

论文原话：

> "Recurrent models typically factor computation along the symbol positions of the input and output sequences. Aligning the positions to steps in computation time, they generate a sequence of hidden states h_t, as a function of the previous hidden state h_{t-1} and the input for position t. This inherently sequential nature precludes parallelization within training examples, which becomes critical at longer sequence lengths."

### 1.2 Attention 的早期使用

在 Transformer 之前，Attention 已经被用于增强 RNN（如 Bahdanau et al. 2014 的 Seq2Seq+Attention）：

```
Seq2Seq + Attention:
  Encoder RNN → [h₁, h₂, ..., hₙ]
  Decoder RNN 每步用 Attention 计算 encoder 输出的加权和
  → Attention 只是"辅助"，核心还是 RNN
```

Transformer 的关键洞察：

> "The Transformer [...] relies entirely on an attention mechanism to draw global dependencies between input and output. The Transformer allows for significantly more parallelization."

**核心创新：把 Attention 从辅助角色提升为主角，完全去掉循环结构。**

## 2. 整体架构：Encoder-Decoder

### 2.1 论文中的架构图（文字重现）

```
                    ┌─────────────────────────┐
                    │      Output Probabilities │
                    │         Softmax            │
                    │         Linear             │
                    └───────────┬───────────────┘
                                │
                    ┌───────────┴───────────────┐
                    │       Add & Norm            │
                    │    Feed Forward (FFN)       │
                    │       Add & Norm            │
                    │   Multi-Head Cross-Attn    │  ← Decoder 特有:
                    │       Add & Norm            │     对 Encoder 输出做 Attention
                    │ Masked Multi-Head Self-Attn │
                    └───────────┬───────────────┘
                                │
            ┌───────────────────┼───────────────────┐
            │                   │                   │  × N (6 layers)
            │   ┌───────────────┴───────────────┐   │
            │   │       Add & Norm                │   │
            │   │    Feed Forward (FFN)           │   │
            │   │       Add & Norm                │   │
            │   │   Multi-Head Self-Attention     │   │ ← Encoder: 只有 Self-Attention
            │   └───────────────┬───────────────┘   │
            │                   │                   │
            └───────────────────┼───────────────────┘
                                │
                    ┌───────────┴───────────────┐
                    │   Input/Output Embedding    │
                    │   + Positional Encoding     │
                    └───────────┬───────────────┘
                                │
                    ┌───────────┴───────────────┐
                    │    Inputs      Outputs      │
                    │  (shifted right)            │
                    └─────────────────────────────┘
```

论文参数 (base model):
- Encoder: 6 layers, Decoder: 6 layers
- d_model = 512
- n_heads = 8
- d_ff = 2048 (FFN inner dimension, 4× d_model)
- d_k = d_v = 64 (d_model / n_heads)
- Total parameters: ~65M

### 2.2 数据流概览

```
输入文本: "I love AI"
   ↓ Tokenizer
Token IDs: [10, 25, 100]
   ↓ Embedding
Embedding: [3, 512] 矩阵 (3 tokens × 512-dim)
   ↓ + Positional Encoding
PE(x):     [3, 512]
   ↓ Encoder (6 layers)
   │  每层: Self-Attention → Add&Norm → FFN → Add&Norm
   ↓
Encoder Output: [3, 512]  ← 源语言表示

Decoder 侧:
   ↓ Embedding + PE
   ↓ Decoder (6 layers)
   │  每层: Masked Self-Attn → Add&Norm
   │         Cross-Attn (Q=Decoder, K/V=Encoder) → Add&Norm
   │         FFN → Add&Norm
   ↓ Linear + Softmax
Output: 下一个 token 的概率分布
```

## 3. Self-Attention — Transformer 的核心

这是整篇论文最重要的部分。让我们从公式出发，逐步建立直觉。

### 3.1 Scaled Dot-Product Attention（论文 Section 3.2.1）

论文中的原始定义：

> "An attention function can be described as mapping a query and a set of key-value pairs to an output, where the query, keys, values, and output are all vectors. The output is computed as a weighted sum of the values, where the weight assigned to each value is computed by a compatibility function of the query with the corresponding key."

**核心公式**:

```
Attention(Q, K, V) = softmax(QK^T / √d_k) V
```

### 3.2 逐元素理解

```python
# 输入: 一个序列的 embedding
x = [x₁, x₂, ..., xₙ]    # 每个 x_i 是 d_model 维向量

# Step 1: 通过线性变换生成 Q, K, V
Q = x @ W_Q    # Query:  "我在找什么?"
K = x @ W_K    # Key:    "我是什么?"
V = x @ W_V    # Value:  "我包含什么信息?"

# 其中 W_Q, W_K, W_V 是 d_model × d_k 的可学习矩阵
# 在 self-attention 中, Q, K, V 都来自同一个输入 x
# 这就是 "self" 的含义

# Step 2: 计算"相关性分数" — Q 和 K 的内积
scores = Q @ K.T    # [n, d_k] @ [d_k, n] → [n, n]
# scores[i][j] = 位置 i 的 Query 对位置 j 的 Key 的"关注程度"

# Step 3: 缩放 (除以 sqrt(d_k))
scores = scores / sqrt(d_k)
# 为什么? 论文的解释:
# "We suspect that for large values of d_k, the dot products grow large
#  in magnitude, pushing the softmax function into regions where it has
#  extremely small gradients."

# Step 4: Softmax — 变成概率分布
attention_weights = softmax(scores, dim=-1)
# 每一行是一个概率分布: 当前位置对所有位置的关注度

# Step 5: 加权聚合 Value
output = attention_weights @ V
# 位置 i 的输出 = sum_j(attention_weights[i][j] × V[j])
```

### 3.3 为什么叫 "Scaled Dot-Product"？

| 变体 | 公式 | 说明 |
|------|------|------|
| Dot-Product | softmax(QK^T) V | 简单，但 d_k 大时 softmax 梯度极小 |
| **Scaled Dot-Product** | softmax(QK^T / √d_k) V | Transformer 使用的，加缩放因子 |
| Additive (Bahdanau) | softmax(v^T tanh(W[Q;K])) V | 前 Transformer 时代的主流 |

论文的实验证明：Dot-Product 更快（可以矩阵乘法），加 √d_k 缩放解决了梯度问题。

### 3.4 Self-Attention 的复杂度分析

```
Q @ K.T:  [n, d_k] @ [d_k, n] → O(n² × d_k)
softmax:  [n, n]              → O(n²)
attn @ V: [n, n] @ [n, d_v]   → O(n² × d_v)

总复杂度: O(n² × d_model)
  n = 序列长度, d_model = 模型维度

当 n 很大时:
  n=512:  512² × 512 ≈ 134M 操作 → OK
  n=4096: 4096² × 512 ≈ 8.6B 操作 → 开始有感觉
  n=128K: 128K² × 512 ≈ 8.4T 操作 → 严重瓶颈!

这就是为什么长上下文推理需要 FlashAttention 等优化。
我们会在后续推理章节中详细讨论。
```

### 3.5 Causal Masking（Decoder 特有）

Decoder 的 Self-Attention 需要 causal mask（也称 look-ahead mask）：

```python
# causal mask: 确保位置 i 只能看到位置 0..i，不能看到 i+1..n
mask = [[0, -∞, -∞, -∞],    # 位置 0 只能看自己
        [0,  0, -∞, -∞],    # 位置 1 可以看 0,1
        [0,  0,  0, -∞],    # 位置 2 可以看 0,1,2
        [0,  0,  0,  0]]    # 位置 3 可以看全部

scores = Q @ K.T / sqrt(d_k) + mask
attention_weights = softmax(scores)
# softmax(-∞) = 0 → 未来位置权重为 0
```

**这对推理至关重要**：causal mask 决定了 Decoder 每生成一个新 token 都只需要看历史 token，这就是 **KV Cache 能工作的原因**。

## 4. Multi-Head Attention（论文 Section 3.2.2）

### 4.1 为什么需要多头？

论文原话：

> "Instead of performing a single attention function [...] we found it beneficial to linearly project the queries, keys and values h times with different, learned linear projections. [...] Multi-head attention allows the model to jointly attend to information from different representation subspaces at different positions."

**直觉理解**：
- 单头：一个词只能以一种方式理解上下文
- 多头：一个词可以同时关注"语法结构"、"语义关联"、"位置模式"等多个维度
- 类比：同一个句子，语言学家看语法、诗人看韵律、翻译家看语义——多头就是让模型同时拥有多个"专家视角"

### 4.2 公式

```
MultiHead(Q, K, V) = Concat(head_1, ..., head_h) W^O

其中:
  head_i = Attention(Q @ W_i^Q, K @ W_i^K, V @ W_i^V)

每个 head 的维度: d_k = d_v = d_model / h
```

关键设计：每个 head 的维度缩小（d_model/h），总计算量与单头（维度 d_model）基本一致。

```
单头 (维度 = d_model):
  Q,K,V ∈ [n, d_model]
  Attention: O(n² × d_model)

多头 (h 个头, 每头维度 = d_model/h):
  每头: Q,K,V ∈ [n, d_model/h]
  每头 Attention: O(n² × d_model/h)
  h 头总和: O(n² × d_model)  ← 相同!
```

### 4.3 完整的代码实现

```python
import torch
import torch.nn as nn
import math

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model=512, n_heads=8):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_k = d_model // n_heads

        # W_Q, W_K, W_V 合并为一个矩阵 → 更高效的实现
        self.W_qkv = nn.Linear(d_model, 3 * d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x, mask=None):
        batch_size, seq_len, _ = x.shape

        # 1. 线性投影: [batch, seq, d_model] → [batch, seq, 3*d_model]
        qkv = self.W_qkv(x)

        # 2. 拆分成 Q, K, V 并 reshape 为多头形式
        q, k, v = qkv.chunk(3, dim=-1)
        q = q.view(batch_size, seq_len, self.n_heads, self.d_k).transpose(1, 2)
        k = k.view(batch_size, seq_len, self.n_heads, self.d_k).transpose(1, 2)
        v = v.view(batch_size, seq_len, self.n_heads, self.d_k).transpose(1, 2)
        # 形状: [batch, n_heads, seq, d_k]

        # 3. Scaled Dot-Product Attention
        scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.d_k)
        # scores: [batch, n_heads, seq, seq]

        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))

        attn_weights = torch.softmax(scores, dim=-1)

        # 4. 加权聚合 V
        out = attn_weights @ v  # [batch, n_heads, seq, d_k]

        # 5. Concat heads + 最终投影
        out = out.transpose(1, 2).contiguous().view(
            batch_size, seq_len, self.d_model)
        out = self.W_o(out)

        return out
```

### 4.4 单头 vs 多头：论文中的消融实验

论文 Table 3 的消融实验：

| 配置 | d_k | PPL (EN-DE) | BLEU (EN-DE) |
|------|-----|-------------|---------------|
| h=8, d_k=64 | 64 | 4.84 | 25.9 |
| h=4, d_k=128 | 128 | 4.80 | 25.7 |
| h=16, d_k=32 | 32 | 4.92 | 25.5 |
| h=1, d_k=512 | 512 | 5.23 | **24.5** |

结论：**单头明显变差（-1.4 BLEU）**，8 头最佳。多头确实学到了不同的表示子空间。

### 4.5 多头在现代 LLM 中的演化：GQA / MQA

这是理解 KV Cache 优化的关键前置知识：

```
MHA (Multi-Head Attention):
  Q heads = K heads = V heads = 8
  每个 head 独立的 K, V → KV Cache 最大

GQA (Grouped-Query Attention) — LLaMA-2/3 使用:
  Q heads = 32, K/V heads = 8 (4个 Q head 共享 1 个 KV head)
  → KV Cache 减少 4×

MQA (Multi-Query Attention) — PaLM 使用:
  Q heads = 32, K/V heads = 1 (所有 Q head 共享同一个 K, V)
  → KV Cache 减少 32×，但质量略有下降
```

```
MHA:                    GQA:                    MQA:
Q: ●●●●●●●●            Q: ●●●●●●●●            Q: ●●●●●●●●
    ││││││││               ││││││││               ││││││││
K: ●●●●●●●●            K: ●  ●                 K: ●
V: ●●●●●●●●            V: ●  ●                 V: ●

KV Cache 大小比:         1/4                     1/32
```

这就是为什么 LLaMA-3 8B 的 KV Cache 可以比同规模 MHA 模型小 4 倍——GQA 是推理优化的重要架构设计。

## 5. Positional Encoding（论文 Section 3.5）

### 5.1 为什么需要位置编码？

Transformer 没有循环结构，Self-Attention 本身是**位置无关**的——对模型来说，"我爱你"和"你爱我"的 attention 模式完全相同（只是 token ID 不同）。

论文原话：

> "Since our model contains no recurrence and no convolution, in order for the model to make use of the order of the sequence, we must inject some information about the relative or absolute position of the tokens in the sequence."

### 5.2 正弦位置编码

论文选择的是 sinusoidal（正弦）位置编码：

```
PE(pos, 2i)   = sin(pos / 10000^(2i/d_model))   — 偶数维度
PE(pos, 2i+1) = cos(pos / 10000^(2i/d_model))   — 奇数维度
```

其中：
- `pos` 是 token 在序列中的位置 (0, 1, 2, ...)
- `i` 是维度的索引 (0, 1, ..., d_model/2 - 1)
- 不同频率的正弦波铺在不同维度上

### 5.3 为什么选正弦？（论文的关键洞察）

> "We chose the sinusoidal version because it may allow the model to extrapolate to sequence lengths longer than the ones encountered during training."

正弦编码有一个优雅的数学性质：**PE(pos+k) 可以表示为 PE(pos) 的线性函数**。

```
sin(pos + k) = sin(pos)cos(k) + cos(pos)sin(k)
cos(pos + k) = cos(pos)cos(k) - sin(pos)sin(k)
```

这意味着模型可能学会利用**相对位置**信息——即使它只见过长度 100 的序列，位置 500 的编码也能通过正弦函数的周期性合理外推。

### 5.4 可视化：不同维度的正弦波

不同维度 i 对应不同频率：

```
i=0 (低频):     ——〰️———————  (缓慢变化, 每个位置都不同)
i=1:            —〰️—〰️—〰️—  (中等频率)
...
i=d_model/2-1 (高频):  〰️〰️〰️〰️〰️〰️〰️  (快速振荡, 相邻位置编码不同)
```

这形成了一个"二进制计数器"式的表示：低频维度记录大致位置（前段/中段/后段），高频维度记录精确位置（相邻 token 的区分）。

### 5.5 实现代码

```python
class PositionalEncoding(nn.Module):
    def __init__(self, d_model=512, max_len=5000):
        super().__init__()

        # 创建位置编码矩阵: [max_len, d_model]
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1).float()  # [max_len, 1]

        # 计算论文公式中的分母: 1 / (10000^(2i/d_model))
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float() *
            (-math.log(10000.0) / d_model)
        )
        # 等价于: 10000^(2i/d_model) 的倒数

        pe[:, 0::2] = torch.sin(position * div_term)   # 偶数位: sin
        pe[:, 1::2] = torch.cos(position * div_term)   # 奇数位: cos
        pe = pe.unsqueeze(0)  # [1, max_len, d_model] — batch 维度

        self.register_buffer('pe', pe)  # 不是可学习参数，但随模型保存

    def forward(self, x):
        # x: [batch, seq_len, d_model]
        return x + self.pe[:, :x.size(1), :]
```

### 5.6 位置编码方案对比

| 方式 | 代表模型 | 优点 | 缺点 |
|------|---------|------|------|
| Sinusoidal | 原始 Transformer, LLaMA-1 | 可外推，无额外参数 | 表达能力有限 |
| Learned | BERT, GPT-2/3, ViT | 灵活适配数据分布 | 不能超训练长度 |
| **RoPE** | LLaMA-2/3, Qwen, Mistral, DeepSeek | 相对位置，天然外推 | 实现稍复杂 |
| ALiBi | BLOOM | 简单偏置项，外推好 | 绝对位置信息弱 |

> **RoPE (Rotary Position Embedding)** 是现代 LLM 的主流选择，通过对 Q 和 K 做旋转变换来注入位置信息。后续在 llama.cpp deep dive 中会详细展开。

## 6. Position-wise Feed-Forward Networks（论文 Section 3.3）

### 6.1 结构

每个 Encoder 和 Decoder 层都包含一个全连接前馈网络，**对每个位置独立应用**：

```
FFN(x) = ReLU(x @ W_1 + b_1) @ W_2 + b_2
```

也就是两个线性变换，中间夹一个 ReLU：

```
Input:  [batch, seq, d_model=512]
  ↓
Linear 1: 512 → 2048  (d_ff = 4 × d_model)
  ↓
ReLU (原始论文) / GELU (GPT) / SiLU (LLaMA)
  ↓
Linear 2: 2048 → 512
  ↓
Output: [batch, seq, d_model=512]
```

### 6.2 为什么叫 "Position-wise"？

对序列中**每个位置独立**做同样的变换。位置 i 的 FFN 只依赖位置 i 的表示，不依赖其他位置。

这形成了一种功能分工：

```
Self-Attention → 负责"跨位置交互" (token 之间传递信息)
FFN           → 负责"位置内变换" (每个 token 独立做知识变换)
```

可以类比为：
- Self-Attention 是"交流层"——每个 token 和其他 token 对话
- FFN 是"思考层"——每个 token 独自消化信息

### 6.3 FFN 在现代 LLM 中的演变

| 模型 | FFN 类型 | 公式 | 说明 |
|------|---------|------|------|
| 原始 Transformer | ReLU | max(0, xW₁)W₂ | 论文使用 |
| GPT-3 | GELU | GELU(xW₁)W₂ | 更平滑的激活 |
| **LLaMA-1/2/3** | **SwiGLU** | (SiLU(xW_gate) ⊙ xW_up) W_down | 门控机制 |
| PaLM | SwiGLU | 同 LLaMA | — |
| Qwen/Mistral | SwiGLU | 同 LLaMA | — |

SwiGLU (Shazeer, 2020) 的理论依据：

```
传统 FFN:        out = Activation(x @ W_1) @ W_2
SwiGLU:         gate = SiLU(x @ W_gate)
                 up   = x @ W_up
                 out  = (gate ⊙ up) @ W_down

门控信号 gate 决定 up 中哪些信息可以通过
→ 类似 LSTM/GRU 的门控思路, 但完全在前馈网络中
→ 在相同计算量下性能显著更好
```

> 现代 LLM 使用 SwiGLU 时，通常调整 intermediate size 使总参数量与传统 FFN 保持一致（SwiGLU 有 3 个权重矩阵 vs 传统 FFN 的 2 个）。

## 7. Add & Norm：残差连接与层归一化（论文 Section 3.1）

### 7.1 残差连接

每个子层（Self-Attention 和 FFN）都包裹在残差连接中：

```
output = LayerNorm(x + Dropout(Sublayer(x)))
```

论文原话：

> "We apply dropout to the output of each sub-layer, before it is added to the sub-layer input and normalized."

残差连接解决了深层网络的梯度消失问题——即使某一层的梯度很小，残差路径（x + ...）也能保证梯度直接回传。

### 7.2 Post-Norm vs Pre-Norm

论文使用的是 **Post-Norm**（先子层，后加残差，再 Norm）：

```
Post-Norm (原始论文):
  x → Sublayer(x) → x + Sublayer(x) → LayerNorm → output

Pre-Norm (现代主流):
  x → LayerNorm(x) → Sublayer(LayerNorm(x)) → x + ... → output
```

为什么现代模型普遍转向 Pre-Norm？

| | Post-Norm | Pre-Norm |
|---|---|---|
| 训练稳定性 | 需要 warmup，梯度可能爆炸 | **不需要 warmup** |
| 收敛速度 | 较慢 | **较快** |
| 最终性能 | 理论上更好（训练充分时） | 实践中几乎一样好 |
| 使用 | 原始 Transformer | GPT-3/4, LLaMA, BLOOM |

### 7.3 实现对比

```python
class EncoderLayer(nn.Module):
    def __init__(self, d_model=512, n_heads=8, d_ff=2048, dropout=0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, n_heads)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Linear(d_ff, d_model),
        )
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        # Post-Norm (论文原版)
        attn_out = self.self_attn(x, mask)
        x = self.norm1(x + self.dropout(attn_out))

        ffn_out = self.ffn(x)
        x = self.norm2(x + self.dropout(ffn_out))
        return x

    def forward_pre_norm(self, x, mask=None):
        # Pre-Norm (现代主流)
        x = x + self.dropout(self.self_attn(self.norm1(x), mask))
        x = x + self.dropout(self.ffn(self.norm2(x)))
        return x
```

### 7.4 现代替代：RMSNorm

LLaMA 系列使用 RMSNorm 替代 LayerNorm：

```
LayerNorm:
  y = (x - mean(x)) / std(x) * γ + β

RMSNorm:
  y = x / RMS(x) * γ
  其中 RMS(x) = sqrt(mean(x²))

省去了减均值的操作 → 更快，且实验证明效果几乎无差异
```

## 8. Decoder 的独特之处

### 8.1 Decoder 的三个子层

Decoder 每层有三个子层（Encoder 只有两个）：

```
Encoder Layer:                    Decoder Layer:
  1. Self-Attention                 1. Masked Self-Attention    ← 多了 "Masked"
  2. FFN                            2. Cross-Attention          ← 多了这个!
                                    3. FFN
```

### 8.2 Masked Self-Attention

Decoder 的第一个子层是 **masked** self-attention——用 causal mask 确保生成位置 i 时只能看到位置 0..i：

```
训练时 (teacher forcing + causal mask):
  输入: "<SOS> 我 爱 你"
  位置 "<SOS>" (0): 只能看 ["<SOS>"]
  位置 "我" (1):    只能看 ["<SOS>", "我"]
  位置 "爱" (2):    只能看 ["<SOS>", "我", "爱"]
  位置 "你" (3):    只能看 ["<SOS>", "我", "爱", "你"]

这样就保证了推理时以自回归方式逐 token 生成。
```

### 8.3 Cross-Attention

Decoder 的第二个子层是 cross-attention——Q 来自 Decoder，K 和 V 来自 Encoder 的输出：

```python
# Decoder 的 cross-attention
Q = decoder_hidden @ W_Q    # 来自 Decoder 自身
K = encoder_output @ W_K    # 来自 Encoder 最终输出!
V = encoder_output @ W_V    # 来自 Encoder 最终输出!

output = Attention(Q, K, V)
# Decoder 的每个位置"关注" Encoder 的全部输出
# 这就是 "cross" 的含义 —— 跨模块的信息交互
```

**Cross-Attention 没有 causal mask**——Decoder 的每个位置都可以看到 Encoder 的**全部**输出。这合理：翻译时，目标语言的每个词都可以参考源语言的完整句子。

### 8.4 Encoder-Decoder → Decoder-Only 的架构演进

```
原始 Transformer (Encoder-Decoder):
  Encoder(源语言) → Decoder(目标语言)
  代表: T5, BART, MarianMT, Whisper
  适用: 翻译、摘要、语音识别

GPT-style (Decoder-Only):
  只有 causal self-attention, 无 cross-attention
  代表: GPT-3/4, LLaMA, Mistral, Qwen, Claude
  适用: 通用语言建模、对话

Encoder-Only:
  只有双向 self-attention (无 causal mask)
  代表: BERT, RoBERTa, DeBERTa
  适用: 文本分类、NER、阅读理解

关键区别:
  • Encoder-Decoder 有 cross-attention → 需要同时运行 Encoder 和 Decoder
  • Decoder-Only 只有 causal self-attention → 统一的自回归范式
  • 现在的 LLM 几乎全是 Decoder-Only
```

**这解释了为什么后续所有推理框架（vLLM、llama.cpp 等）都只关注 Decoder-Only 架构**——Encoder 只在 prefill 时跑一次，Decoder 要逐 token 跑 N 次，优化的收益集中在 Decoder 上。

## 9. 训练细节（论文 Section 5）

### 9.1 数据集和硬件

| 项目 | 详情 |
|------|------|
| 训练数据 | WMT 2014 English-German (~4.5M 句对) |
| | WMT 2014 English-French (~36M 句对) |
| 硬件 | 8 × NVIDIA P100 GPU |
| 训练时间 | Base: 12h (100K steps), Big: 3.5 days (300K steps) |
| Tokenization | Byte-Pair Encoding (BPE) |
| | EN-DE: 37K shared vocab, EN-FR: 32K vocab |

### 9.2 Optimizer: Adam + Custom LR Schedule

论文使用 Adam (beta1=0.9, beta2=0.98, epsilon=10^-9)，配合自定的学习率调度：

```
lrate = d_model^(-0.5) * min(step_num^(-0.5), step_num * warmup_steps^(-1.5))
```

```python
def transformer_lr_schedule(step, d_model=512, warmup_steps=4000):
    """论文公式 (3) -- 带 warmup 的衰减学习率"""
    arg1 = step ** (-0.5)              # 衰减阶段
    arg2 = step * warmup_steps ** (-1.5)  # warmup 阶段 (线性增长)
    return (d_model ** (-0.5)) * min(arg1, arg2)
```

学习率曲线：

```
LR
^
|      /|
|     / |
|    /  |
|   /   |_____
|  /          \_____  (step^(-0.5) 衰减)
| /                  \
+-------------------------> steps
   warmup (4000 steps)
```

**为什么需要 warmup？** 论文使用 Post-Norm，训练初期梯度不稳定。Warmup 给模型一个"缓冲期"来逐渐增加学习率，避免早期梯度爆炸。现代模型用 Pre-Norm 后，warmup 的重要性大大降低。

### 9.3 Regularization

论文使用了三种正则化手段：

| 方法 | 应用位置 | 参数 |
|------|---------|------|
| **Dropout** | 每个子层的输出 + Embedding + PE | rate=0.1 |
| **Label Smoothing** | 训练目标 (交叉熵) | epsilon_ls=0.1 |
| **Residual Dropout** | 残差连接加法前 | rate=0.1 |

论文关于 Label Smoothing 的有趣发现：

> "Label smoothing improves accuracy and BLEU score, but harms perplexity."

这是 trade-off：Label Smoothing 让模型不那么"自信"，生成任务（BLEU）更好，但困惑度反而变差——因为模型被迫给错误答案分配概率。

## 10. 实验结果（论文 Section 6）

### 10.1 关键结果

| 模型 | EN-DE BLEU | EN-FR BLEU | 训练成本 (FLOPs) |
|------|-----------|-----------|-----------------|
| Transformer (Base) | 27.3 | 38.1 | 3.3e18 |
| Transformer (Big) | **28.4** | **41.0** | 2.3e19 |
| 之前 SOTA (ConvS2S) | 26.4 | 40.5 | 9.6e18 |
| 之前 SOTA (MoE) | 26.0 | 40.6 | 2.0e20 |

Transformer Big 在 EN-DE 上超过所有之前模型，且**训练成本只是 ConvS2S 的 1/4，MoE 的 1/10**。

### 10.2 消融实验（论文 Table 3）

| 变体 | BLEU | 变化 | 说明 |
|------|------|------|------|
| Base (h=8, d_k=64) | 25.9 | -- | 基准 |
| h=16, d_k=32 | 25.5 | -0.4 | 头太多，每头太小 |
| h=1, d_k=512 | 24.5 | **-1.4** | **单头显著变差** |
| d_k=16 (h 不变) | 25.6 | -0.3 | d_k 缩小影响较小 |

**核心结论**：多头是关键（h=1 掉 1.4 BLEU），头数和 d_k 需要平衡。

## 11. 从 Transformer 论文到现代 LLM 推理

### 11.1 论文中的每个概念如何影响推理

| 论文概念 | 在推理框架中的直接对应 |
|---------|---------------------|
| Self-Attention (QK^T / sqrt(d_k)) | FlashAttention tiling 策略、PagedAttention block 计算 |
| Softmax over K dimension | Online Softmax (FlashAttention 的分块技术) |
| Causal Mask (下三角) | **KV Cache 的工作前提** -- 只有历史的 K,V 需要缓存 |
| K, V 矩阵 | **KV Cache 本身** -- 缓存 Decoder 每层的 K 和 V |
| Multi-Head (h heads x d_k) | GQA/MQA 减少 KV head 数来压缩 KV Cache |
| d_ff = 4 x d_model | FFN 占推理 FLOPs 的约 2/3，权重大头在 FFN |
| Residual + LayerNorm | kernel fusion 机会（残差+Norm 融合为单个 kernel） |
| Positional Encoding | RoPE 在 Attention 中插入旋转变换，影响 kernel 复杂度 |
| Encoder vs Decoder | Encoder 只跑 1 次（prefill），Decoder 跑 N 次 -- 优化重点是 Decoder |

### 11.2 推理中的计算量分布

以 LLaMA-7B (Decoder-Only, d_model=4096, n_layers=32) 生成一个 token 为例：

```
单个 Decode Step 的 FLOPs 分解:
  Attention QKV 投影: 3 x (4096 x 4096) x 1 token  ~ 100M FLOPs
  Attention Score:    1 x (seq_len x 4096)          ~ 4M x seq_len
  Attention Output:   (4096 x 4096) x 1 token       ~ 33M FLOPs
  FFN up:             (4096 x 11008) x 1 token      ~ 90M FLOPs
  FFN down:           (11008 x 4096) x 1 token      ~ 90M FLOPs
  Output projection:  (4096 x 32000) x 1 token      ~ 260M FLOPs

  总计 (seq_len=4096): ~600M FLOPs per token

关键观察:
  Attention 的 FLOPs 随 seq_len 线性增长 (要读全部 KV Cache)
  FFN 的 FLOPs 与 seq_len 无关 (只对当前 token)
  --> 长序列时 Attention 成为计算瓶颈! 这就是长上下文需要优化的原因
```

### 11.3 现代 LLM 对原始 Transformer 的改动汇总

| 改动 | 原始 | 现代 | 推理影响 |
|------|------|------|---------|
| 架构 | Encoder-Decoder | **Decoder-Only** | 去掉 cross-attn，推理更简单 |
| Norm 位置 | Post-Norm | **Pre-Norm** | 训练稳定，推理无影响 |
| Norm 类型 | LayerNorm | **RMSNorm** | 推理时少一次减均值 |
| 激活函数 | ReLU | **SwiGLU/SiLU** | FFN 有 3 个矩阵，参数量受控 |
| 位置编码 | Sinusoidal | **RoPE** | 在 attention kernel 做旋转变换 |
| Attention | MHA | **GQA (LLaMA-2/3)** | KV Cache 减少 4-8x |
| 词表大小 | 32-37K | 32K-128K | 输出投影矩阵更大 |

### 11.4 阅读路径

1. **Transformer 推理视角** -- Prefill/Decode 阶段、KV Cache 精确计算
2. **量化基础** -- GGUF/AWQ/GPTQ/FP8
3. **GPU 显存布局** -- 多模态 / 工具调用的显存冲击
4. 然后进入各框架的 deep dive

## 12. 代码实验：从零实现一个 Mini Decoder-Only Transformer

下面是一个最小但完整的 Transformer 实现，约 120 行代码，包含所有核心组件。可以直接运行验证。

In [1]:
# 最小完整 Transformer 实现 (Decoder-Only, 类似 GPT)
# 基于 "Attention Is All You Need" 的所有核心组件

import torch
import torch.nn as nn
import math

# --- Positional Encoding (Sinusoidal, 论文 Section 3.5) ---
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1).float()
        div_term = torch.exp(torch.arange(0, d_model, 2).float() *
                            (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1), :]

# --- Multi-Head Self-Attention (论文 Section 3.2.2) ---
class MultiHeadSelfAttention(nn.Module):
    def __init__(self, d_model=512, n_heads=8, dropout=0.1):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.d_k = d_model // n_heads

        self.W_qkv = nn.Linear(d_model, 3 * d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, causal_mask=True):
        B, T, C = x.shape

        qkv = self.W_qkv(x)  # [B, T, 3*C]
        q, k, v = qkv.chunk(3, dim=-1)
        q = q.view(B, T, self.n_heads, self.d_k).transpose(1, 2)  # [B, H, T, d_k]
        k = k.view(B, T, self.n_heads, self.d_k).transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.d_k).transpose(1, 2)

        # Scaled Dot-Product Attention
        scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.d_k)

        if causal_mask:
            mask = torch.triu(torch.ones(T, T, device=x.device), diagonal=1).bool()
            scores = scores.masked_fill(mask, float('-inf'))

        attn = torch.softmax(scores, dim=-1)
        attn = self.dropout(attn)

        out = attn @ v  # [B, H, T, d_k]
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        return self.W_o(out)

# --- Feed-Forward (论文 Section 3.3) ---
class FeedForward(nn.Module):
    def __init__(self, d_model=512, d_ff=2048, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
        )

    def forward(self, x):
        return self.net(x)

# --- Decoder Layer ---
class DecoderLayer(nn.Module):
    def __init__(self, d_model=512, n_heads=8, d_ff=2048, dropout=0.1):
        super().__init__()
        self.self_attn = MultiHeadSelfAttention(d_model, n_heads, dropout)
        self.ffn = FeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # Post-Norm (原始论文风格)
        x = x + self.dropout(self.self_attn(self.norm1(x)))
        x = x + self.dropout(self.ffn(self.norm2(x)))
        return x

# --- 完整 MiniGPT 模型 ---
class MiniGPT(nn.Module):
    def __init__(self, vocab_size, d_model=512, n_heads=8, n_layers=6,
                 d_ff=2048, max_len=1024, dropout=0.1):
        super().__init__()
        self.token_embed = nn.Embedding(vocab_size, d_model)
        self.pos_encoding = PositionalEncoding(d_model, max_len)
        self.layers = nn.ModuleList([
            DecoderLayer(d_model, n_heads, d_ff, dropout)
            for _ in range(n_layers)
        ])
        self.ln_final = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)

        # Weight tying: embedding 和 lm_head 共享权重
        self.lm_head.weight = self.token_embed.weight

        self.apply(self._init_weights)
        print(f"Model params: {sum(p.numel() for p in self.parameters()):,}")

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.xavier_uniform_(module.weight)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)

    def forward(self, idx):
        B, T = idx.shape
        # 论文: "we multiply the embedding by sqrt(d_model)"
        x = self.token_embed(idx) * math.sqrt(self.token_embed.embedding_dim)
        x = self.pos_encoding(x)
        for layer in self.layers:
            x = layer(x)
        x = self.ln_final(x)
        logits = self.lm_head(x)
        return logits

# --- 测试 ---
print("=" * 60)
print("测试 MiniGPT (Decoder-Only Transformer)")
print("=" * 60)

model = MiniGPT(vocab_size=1000, d_model=256, n_heads=4, n_layers=3, d_ff=1024)

x = torch.randint(0, 1000, (2, 10))
logits = model(x)

print(f"\n输入形状: {x.shape}")
print(f"输出形状: {logits.shape}  (应为 [2, 10, 1000])")

print(f"\n架构验证:")
print(f"  Embedding       -> {model.token_embed}")
print(f"  Positional Enc  -> Sinusoidal (PE table: {model.pos_encoding.pe.shape})")
print(f"  Decoder Layers  -> {len(model.layers)} layers")
print(f"  Final LayerNorm -> {model.ln_final}")
print(f"  LM Head         -> {model.lm_head}")
print(f"  Weight Tying    -> {model.token_embed.weight is model.lm_head.weight}")
print(f"\n与原始论文的对应关系:")
print(f"  MultiHeadSelfAttention <- 论文 Section 3.2.2")
print(f"  PositionalEncoding      <- 论文 Section 3.5")
print(f"  FeedForward (ReLU)      <- 论文 Section 3.3")
print(f"  LayerNorm + Residual    <- 论文 Section 3.1")
print(f"  Causal Mask             <- 论文 Section 3.2.3 (Decoder)")
print(f"  Embed x sqrt(d_model)  <- 论文脚注")

ModuleNotFoundError: No module named 'torch'

## 13. 论文关键公式速查

| 公式 | 论文编号 | 含义 |
|------|---------|------|
| Attention(Q,K,V) = softmax(QK^T / sqrt(d_k)) V | (1) | Scaled Dot-Product Attention |
| MultiHead(Q,K,V) = Concat(head_1,...,head_h) W^O | (2) | Multi-Head Attention |
| FFN(x) = max(0, xW1+b1)W2+b2 | (2) | Position-wise FFN |
| PE(pos,2i) = sin(pos/10000^(2i/d_model)) | -- | 位置编码 (偶数维) |
| PE(pos,2i+1) = cos(pos/10000^(2i/d_model)) | -- | 位置编码 (奇数维) |
| lrate = d^(-0.5) * min(...) | (3) | 学习率调度 |

## 14. 自测问题

1. **Self-Attention 中为什么要除以 sqrt(d_k)？**
   → d_k 很大时 QK^T 的值太大，导致 softmax 梯度消失

2. **为什么 Multi-Head 比 Single-Head 好？**
   → 不同 head 关注不同的表示子空间（语法/语义/位置等）

3. **Transformer 如何捕捉位置信息？**
   → Sinusoidal Positional Encoding / RoPE / ALiBi

4. **Post-Norm 和 Pre-Norm 的区别？**
   → Post-Norm 需要 warmup；Pre-Norm 训练更稳定，是现代主流

5. **Encoder vs Decoder 的关键区别？**
   → Decoder 有 causal mask + cross-attention；Encoder 没有

6. **为什么现在都用 Decoder-Only？**
   → 架构更简单，自回归范式统一，推理时不需要跑 Encoder

7. **GQA 相比 MHA 的优势？**
   → 多个 Q head 共享 KV head → KV Cache 减少 4-8x

## 参考资料

- [Attention Is All You Need](https://arxiv.org/abs/1706.03762) — 原始论文
- [The Illustrated Transformer](https://jalammar.github.io/illustrated-transformer/) — Jay Alammar 图解版
- [The Annotated Transformer](http://nlp.seas.harvard.edu/annotated-transformer/) — Harvard NLP 逐行代码解读
- [Let's build GPT from scratch](https://www.youtube.com/watch?v=kCc8FmEb1nY) — Andrej Karpathy 视频讲解
- 本项目的下一篇: `01-transformer-inference-primer.ipynb` — 推理视角的 Transformer